Task 1

In [ ]:
import math
import matplotlib.pyplot as plt
import networkx as nx

locations = {
    "Receiving_Area": (0, 0),
    "Storage_A": (2, 1),
    "Storage_B": (1, 4),
    "Sorting_Area": (4, 2),
    "Inspection_Area": (5, 5),
    "Packing_Station": (7, 6),
}

start_node = "Receiving_Area"
goal_node = "Packing_Station"

edges = [
    ("Receiving_Area", "Storage_B", 4.1),
    ("Receiving_Area", "Storage_A", 2.2),
    ("Storage_B", "Inspection_Area", 5.0),
    ("Storage_B", "Sorting_Area", 6.0),
    ("Storage_A", "Sorting_Area", 2.2),
    ("Sorting_Area", "Inspection_Area", 3.2),
    ("Sorting_Area", "Packing_Station", 5.0),
    ("Inspection_Area", "Packing_Station", 2.2),
]

G = nx.DiGraph()
for node, pos in locations.items():
  G.add_node(node, pos=pos)

for u, v, weight in edges:
  G.add_edge(u, v, weight=weight)


def heuristic(current, goal=goal_node):
  x1, y1 = locations[current]
  x2, y2 = locations[goal]
  return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


heuristic_values = {
    node: round(heuristic(node, goal_node), 2) for node in locations
}

print("=== Heuristic Values (Euclidean Distance to Packing_Station) ===")
for node, h_val in heuristic_values.items():
  print(f"{node:<20} : {h_val:.2f} km")

print("\n=== Verification of Heuristic Consistency Condition ===")
print(
    f"{'Edge (n -> m)':<35} | {'h(n)':<6} | {'cost(n,m)':<10} | {'h(m)':<6} |"
    f" {'cost+h(m)':<10} | {'Satisfied?'}"
)
print("-" * 85)

for u, v, weight in edges:
  hn = heuristic(u, goal_node)
  hm = heuristic(v, goal_node)
  condition_met = hn <= (weight + hm + 1e-9)
  print(
      f"{u + ' -> ' + v:<35} | {hn:<6.2f} | {weight:<10.1f} | {hm:<6.2f} |"
      f" {weight + hm:<10.2f} | {condition_met}"
  )

plt.figure(figsize=(10, 6))

pos = locations

node_colors = []
for node in G.nodes():
  if node == start_node:
    node_colors.append("lightcoral")
  elif node == goal_node:
    node_colors.append("gold")
  else:
    node_colors.append("lightblue")

nx.draw_networkx_nodes(G, pos, node_size=2000, node_color=node_colors)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")
nx.draw_networkx_edges(
    G, pos, edgelist=G.edges(), arrowstyle="->", arrowsize=20, edge_color="gray"
)

edge_labels = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=10)

plt.title("Warehouse Network Graph Representation", fontsize=14, fontweight="bold")
plt.xlabel("X Coordinates")
plt.ylabel("Y Coordinates")
plt.grid(True, linestyle="--", alpha=0.5)
plt.axis("on")
plt.tight_layout()
plt.show()

Task 2

In [ ]:
import heapq
import math
import matplotlib.pyplot as plt
import networkx as nx

# ---------------------------------------------------------
# 1. Airport Locations and Coordinates
# ---------------------------------------------------------
locations = {
    "Baggage_Area": (0, 0),
    "Security": (2, 1),
    "Checkpoint": (1, 4),
    "Food_Court": (4, 2),
    "Terminal_Hall": (5, 5),
    "Departure_Gate": (8, 6),
}

start_node = "Baggage_Area"
goal_node = "Departure_Gate"

# ---------------------------------------------------------
# 2. Weighted Graph Representation
# ---------------------------------------------------------
edges = [
    ("Baggage_Area", "Checkpoint", 4.1),
    ("Baggage_Area", "Security", 2.2),
    ("Checkpoint", "Terminal_Hall", 5.0),
    ("Security", "Food_Court", 2.2),
    ("Food_Court", "Terminal_Hall", 3.2),
    ("Food_Court", "Departure_Gate", 6.0),
    ("Terminal_Hall", "Departure_Gate", 3.2),
]

G = nx.DiGraph()
for node, pos in locations.items():
  G.add_node(node, pos=pos)

for u, v, weight in edges:
  G.add_edge(u, v, weight=weight)


# ---------------------------------------------------------
# 3. Euclidean Distance Heuristic: h(n)
# ---------------------------------------------------------
def heuristic(node, goal=goal_node):
  x1, y1 = locations[node]
  x2, y2 = locations[goal]
  return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


# ---------------------------------------------------------
# 4. Greedy Best-First Search (GBFS) Implementation
# ---------------------------------------------------------
def greedy_best_first_search(graph, start, goal):
  pq = []
  count = 0
  # Priority Queue Item: (h(n), counter, current_node, path, total_g_cost)
  heapq.heappush(pq, (heuristic(start), count, start, [start], 0.0))

  visited = set()
  expansion_sequence = []

  while pq:
    h_val, _, current, path, total_cost = heapq.heappop(pq)

    if current in visited:
      continue

    visited.add(current)
    expansion_sequence.append(current)

    if current == goal:
      return expansion_sequence, path, total_cost

    for neighbor in graph.neighbors(current):
      if neighbor not in visited:
        edge_cost = graph[current][neighbor]["weight"]
        count += 1
        heapq.heappush(
            pq,
            (
                heuristic(neighbor),
                count,
                neighbor,
                path + [neighbor],
                total_cost + edge_cost,
            ),
        )

  return expansion_sequence, [], 0.0


# Execute GBFS
expansion_seq, solution_path, path_cost = greedy_best_first_search(
    G, start_node, goal_node
)

# Output Results
print("=== GBFS Execution Results ===")
print("Node Expansion Sequence :", " -> ".join(expansion_seq))
print("Solution Path           :", " -> ".join(solution_path))
print(f"Total Path Cost         : {path_cost:.2f} km")

# ---------------------------------------------------------
# 5. NetworkX Graph Visualization with Highlighted Path
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
pos = locations

# Highlight edges in solution path
path_edges = list(zip(solution_path[:-1], solution_path[1:]))

# Node color assignment
node_colors = []
for node in G.nodes():
  if node == start_node:
    node_colors.append("lightcoral")
  elif node == goal_node:
    node_colors.append("gold")
  elif node in solution_path:
    node_colors.append("lightgreen")
  else:
    node_colors.append("lightblue")

# Draw graph elements
nx.draw_networkx_nodes(G, pos, node_size=2200, node_color=node_colors)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")

# Draw background edges
nx.draw_networkx_edges(
    G, pos, edgelist=G.edges(), arrowstyle="->", arrowsize=20, edge_color="gray"
)

# Draw solution path edges
nx.draw_networkx_edges(
    G,
    pos,
    edgelist=path_edges,
    arrowstyle="->",
    arrowsize=25,
    edge_color="green",
    width=3,
)

# Edge weight labels
edge_labels = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=10)

plt.title(
    "Airport Baggage Handling - GBFS Solution Path",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("X Coordinates")
plt.ylabel("Y Coordinates")
plt.grid(True, linestyle="--", alpha=0.5)
plt.axis("on")
plt.tight_layout()
plt.show()

Task 3

In [ ]:
import heapq
import math
import matplotlib.pyplot as plt
import networkx as nx

# ---------------------------------------------------------
# 1. Hospital Locations and Coordinates
# ---------------------------------------------------------
locations = {
    "Pharmacy": (0, 0),
    "Main_Corridor": (2, 1),
    "Patient_Wing": (1, 4),
    "Nursing_Station": (4, 2),
    "Laboratory": (5, 5),
    "Emergency_Ward": (8, 6),
}

start_node = "Pharmacy"
goal_node = "Emergency_Ward"

# ---------------------------------------------------------
# 2. Corridor Graph Representation (Weighted Directed Graph)
# ---------------------------------------------------------
edges = [
    ("Pharmacy", "Patient_Wing", 4.1),
    ("Pharmacy", "Main_Corridor", 2.2),
    ("Patient_Wing", "Laboratory", 5.0),
    ("Main_Corridor", "Nursing_Station", 2.2),
    ("Nursing_Station", "Laboratory", 3.2),
    ("Nursing_Station", "Emergency_Ward", 6.0),
    ("Laboratory", "Emergency_Ward", 3.2),
]

G = nx.DiGraph()
for node, pos in locations.items():
  G.add_node(node, pos=pos)

for u, v, weight in edges:
  G.add_edge(u, v, weight=weight)


# ---------------------------------------------------------
# 3. Euclidean Distance Heuristic: h(n)
# ---------------------------------------------------------
def heuristic(node, goal=goal_node):
  x1, y1 = locations[node]
  x2, y2 = locations[goal]
  return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


# ---------------------------------------------------------
# 4. A* Search Implementation: f(n) = g(n) + h(n)
# ---------------------------------------------------------
def a_star_search(graph, start, goal):
  pq = []
  count = 0

  g_costs = {node: float("inf") for node in graph.nodes()}
  g_costs[start] = 0.0

  f_costs = {node: float("inf") for node in graph.nodes()}
  f_costs[start] = heuristic(start, goal)

  # Priority Queue Item: (f(n), counter, g(n), node, path)
  heapq.heappush(pq, (f_costs[start], count, 0.0, start, [start]))

  visited = set()
  expansion_sequence = []
  node_metrics = []

  while pq:
    f, _, g, current, path = heapq.heappop(pq)

    if current in visited:
      continue

    visited.add(current)
    expansion_sequence.append(current)
    node_metrics.append((current, g, heuristic(current, goal), f))

    if current == goal:
      return expansion_sequence, path, g, node_metrics

    for neighbor in graph.neighbors(current):
      weight = graph[current][neighbor]["weight"]
      tentative_g = g + weight

      if tentative_g < g_costs[neighbor]:
        g_costs[neighbor] = tentative_g
        h_val = heuristic(neighbor, goal)
        f_costs[neighbor] = tentative_g + h_val
        count += 1
        heapq.heappush(
            pq, (f_costs[neighbor], count, tentative_g, neighbor, path + [neighbor])
        )

  return expansion_sequence, [], float("inf"), node_metrics


# Execute A* Search
expansion_seq, solution_path, total_cost, selected_node_metrics = (
    a_star_search(G, start_node, goal_node)
)

# Display Execution Results
print("=== A* Search Execution Results ===")
print("Node Expansion Sequence :", " -> ".join(expansion_seq))
print("Solution Path           :", " -> ".join(solution_path))
print(f"Total Path Cost         : {total_cost:.2f} km")

# ---------------------------------------------------------
# 5. NetworkX Graph Visualization with Highlighted Path
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
pos = locations

path_edges = list(zip(solution_path[:-1], solution_path[1:]))

node_colors = []
for node in G.nodes():
  if node == start_node:
    node_colors.append("lightcoral")
  elif node == goal_node:
    node_colors.append("gold")
  elif node in solution_path:
    node_colors.append("lightgreen")
  else:
    node_colors.append("lightblue")

nx.draw_networkx_nodes(G, pos, node_size=2200, node_color=node_colors)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")

nx.draw_networkx_edges(
    G, pos, edgelist=G.edges(), arrowstyle="->", arrowsize=20, edge_color="gray"
)

nx.draw_networkx_edges(
    G,
    pos,
    edgelist=path_edges,
    arrowstyle="->",
    arrowsize=25,
    edge_color="green",
    width=3,
)

edge_labels = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=10)

plt.title(
    "Emergency Supply Robot - A* Search Solution Path",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("X Coordinates")
plt.ylabel("Y Coordinates")
plt.grid(True, linestyle="--", alpha=0.5)
plt.axis("on")
plt.tight_layout()
plt.show()

Task 4

In [ ]:
import heapq
import math
import matplotlib.pyplot as plt
import networkx as nx

# ---------------------------------------------------------
# 1. Locations and Coordinates
# ---------------------------------------------------------
locations = {
    "Distribution_Center": (0, 0),
    "Zone_A": (2, 1),
    "Zone_B": (1, 4),
    "Zone_C": (4, 2),
    "Zone_D": (5, 5),
    "Customer_Building": (8, 6),
}

start_node = "Distribution_Center"
goal_node = "Customer_Building"

# ---------------------------------------------------------
# 2. Weighted Graph Representation
# ---------------------------------------------------------
edges = [
    ("Distribution_Center", "Zone_B", 3.0),
    ("Distribution_Center", "Zone_A", 2.2),
    ("Zone_B", "Zone_D", 5.0),
    ("Zone_A", "Zone_C", 2.2),
    ("Zone_C", "Zone_D", 3.2),
    ("Zone_C", "Customer_Building", 6.0),
    ("Zone_D", "Customer_Building", 3.2),
]

G = nx.DiGraph()
for node, pos in locations.items():
  G.add_node(node, pos=pos)

for u, v, weight in edges:
  G.add_edge(u, v, weight=weight)


# ---------------------------------------------------------
# 3. Euclidean Distance Heuristic
# ---------------------------------------------------------
def heuristic(node, goal=goal_node):
  x1, y1 = locations[node]
  x2, y2 = locations[goal]
  return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


# ---------------------------------------------------------
# 4. Weighted A* Search Implementation: f(n) = g(n) + w * h(n)
# ---------------------------------------------------------
def weighted_a_star(graph, start, goal, w):
  pq = []
  count = 0

  g_costs = {node: float("inf") for node in graph.nodes()}
  g_costs[start] = 0.0

  f_costs = {node: float("inf") for node in graph.nodes()}
  f_costs[start] = w * heuristic(start, goal)

  heapq.heappush(pq, (f_costs[start], count, 0.0, start, [start]))

  visited = set()
  expansion_sequence = []

  while pq:
    f, _, g, current, path = heapq.heappop(pq)

    if current in visited:
      continue

    visited.add(current)
    expansion_sequence.append(current)

    if current == goal:
      return expansion_sequence, path, g, len(expansion_sequence)

    for neighbor in graph.neighbors(current):
      weight = graph[current][neighbor]["weight"]
      tentative_g = g + weight

      if tentative_g < g_costs[neighbor]:
        g_costs[neighbor] = tentative_g
        h_val = heuristic(neighbor, goal)
        f_costs[neighbor] = tentative_g + (w * h_val)
        count += 1
        heapq.heappush(
            pq, (f_costs[neighbor], count, tentative_g, neighbor, path + [neighbor])
        )

  return expansion_sequence, [], float("inf"), len(expansion_sequence)


# Run experiment for w = 1.0, 1.5, 2.0, 3.0
weights = [1.0, 1.5, 2.0, 3.0]
results = []

print("=== Weighted A* Experimental Results ===")
for w in weights:
  seq, path, cost, nodes_exp = weighted_a_star(G, start_node, goal_node, w)
  results.append((w, " -> ".join(path), round(cost, 2), nodes_exp))
  print(
      f"Weight w = {w:<3} | Path: {' -> '.join(path):<55} | Cost: {cost:.2f} km"
      f" | Nodes Expanded: {nodes_exp}"
  )

# ---------------------------------------------------------
# 5. NetworkX Graph Visualization (Normal A* vs Greedy Weighted A*)
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
pos = locations

_, opt_path, _, _ = weighted_a_star(G, start_node, goal_node, 1.0)
opt_edges = list(zip(opt_path[:-1], opt_path[1:]))

node_colors = [
    "lightcoral"
    if n == start_node
    else "gold"
    if n == goal_node
    else "lightgreen"
    if n in opt_path
    else "lightblue"
    for n in G.nodes()
]

nx.draw_networkx_nodes(G, pos, node_size=2200, node_color=node_colors)
nx.draw_networkx_labels(G, pos, font_size=8, font_weight="bold")
nx.draw_networkx_edges(
    G, pos, edgelist=G.edges(), arrowstyle="->", arrowsize=20, edge_color="gray"
)
nx.draw_networkx_edges(
    G,
    pos,
    edgelist=opt_edges,
    arrowstyle="->",
    arrowsize=25,
    edge_color="green",
    width=3,
)

edge_labels = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=10)

plt.title(
    "Delivery Drone - Weighted A* Graph Representation (Optimal Path Highlighted)",
    fontsize=12,
    fontweight="bold",
)
plt.xlabel("X Coordinates")
plt.ylabel("Y Coordinates")
plt.grid(True, linestyle="--", alpha=0.5)
plt.axis("on")
plt.tight_layout()
plt.show()